# Scoring + Evaluation of Google Search Trend

In [ ]:
!pip install lightgbm
!pip install scikit-learn
!pip install pandas numpy

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [ ]:
all_stars_19_20 = [
    # East
    "Kemba Walker",
    "Trae Young",
    "Giannis Antetokounmpo",
    "Pascal Siakam",
    "Joel Embiid",
    "Kyle Lowry",
    "Ben Simmons",
    "Jimmy Butler",
    "Khris Middleton",
    "Bam Adebayo",
    "Jayson Tatum",
    "Domantas Sabonis",
    # West
    "James Harden",
    "Luka Dončić",
    "LeBron James",
    "Kawhi Leonard",
    "Anthony Davis",
    "Chris Paul",
    "Russell Westbrook",
    "Damian Lillard",
    "Donovan Mitchell",
    "Brandon Ingram",
    "Nikola Jokić",
    "Rudy Gobert",
    "Devin Booker",
]

all_stars_20_21 = [
    # East
    "Bradley Beal",
    "Kyrie Irving",
    "Giannis Antetokounmpo",
    "Kevin Durant",
    "Joel Embiid",
    "Jaylen Brown",
    "James Harden",
    "Zach LaVine",
    "Ben Simmons",
    "Julius Randle",
    "Domantas Sabonis",
    "Jayson Tatum",
    "Nikola Vučević",
    # West
    "Stephen Curry",
    "Luka Dončić",
    "LeBron James",
    "Kawhi Leonard",
    "Nikola Jokić",
    "Devin Booker",
    "Mike Conley",
    "Damian Lillard",
    "Donovan Mitchell",
    "Chris Paul",
    "Anthony Davis",
    "Paul George",
    "Zion Williamson",
    "Rudy Gobert",
]

all_stars_21_22 = [
    # East
    "Giannis Antetokounmpo",
    "Kevin Durant",
    "Joel Embiid",
    "DeMar DeRozan",
    "Trae Young",
    "Jimmy Butler",
    "Darius Garland",
    "James Harden",
    "Zach LaVine",
    "Fred VanVleet",
    "Khris Middleton",
    "Jayson Tatum",
    "LaMelo Ball",
    "Jarrett Allen",
    # West
    "LeBron James",
    "Nikola Jokić",
    "Stephen Curry",
    "Ja Morant",
    "Andrew Wiggins",
    "Devin Booker",
    "Luka Dončić",
    "Donovan Mitchell",
    "Chris Paul",
    "Draymond Green",
    "Rudy Gobert",
    "Karl-Anthony Towns",
    "Dejounte Murray",
]

all_stars_22_23 = [
    # East
    "Kyrie Irving",
    "Donovan Mitchell",
    "Giannis Antetokounmpo",
    "Kevin Durant",
    "Jayson Tatum",
    "Bam Adebayo",
    "Jaylen Brown",
    "DeMar DeRozan",
    "Tyrese Haliburton",
    "Jrue Holiday",
    "Julius Randle",
    "Joel Embiid",
    "Pascal Siakam",
    # West
    "Stephen Curry",
    "Luka Dončić",
    "LeBron James",
    "Zion Williamson",
    "Nikola Jokić",
    "Paul George",
    "Shai Gilgeous-Alexander",
    "Jaren Jackson Jr.",
    "Damian Lillard",
    "Lauri Markkanen",
    "Ja Morant",
    "Domantas Sabonis",
    "Anthony Edwards",
    "De'Aaron Fox",
]

all_stars_23_24 = [
    # East
    "Giannis Antetokounmpo",
    "Jayson Tatum",
    "Joel Embiid",
    "Damian Lillard",
    "Tyrese Haliburton",
    "Bam Adebayo",
    "Paolo Banchero",
    "Jaylen Brown",
    "Jalen Brunson",
    "Tyrese Maxey",
    "Donovan Mitchell",
    "Julius Randle",
    "Trae Young",
    "Scottie Barnes",
    # West
    "Luka Dončić",
    "Shai Gilgeous-Alexander",
    "LeBron James",
    "Kevin Durant",
    "Nikola Jokić",
    "Devin Booker",
    "Stephen Curry",
    "Anthony Davis",
    "Anthony Edwards",
    "Paul George",
    "Kawhi Leonard",
    "Karl-Anthony Towns",
]

In [ ]:
all_stars_lists = {
    "2019-2020": all_stars_19_20,
    "2020-2021": all_stars_20_21,
    "2021-2022": all_stars_21_22,
    "2022-2023": all_stars_22_23,
    "2023-2024": all_stars_23_24,
}

### Step 1: Get a score for each player

1. Extract features from dataset:


*   trends_mean: average search interest over time
*   trends_last30 = average search interest in last 30 days before selection
*   trends_max = highest value (peak) for a player
*   trends_slope = Slope of linear regression line, which indicates a player's trend direction and strength.

2.   Normalize and score: Each player's features are normalized, weights are defined, and combined to calculate a final score.






In [ ]:
def extract_features(series, last_n=30):
    s = series.values
    n = len(s)

    trends_mean = np.mean(s)
    trends_last30 = np.mean(s[-last_n:]) if n >= last_n else np.mean(s)
    trends_max = np.max(s)
    X = np.arange(len(s)).reshape(-1, 1)
    lr = LinearRegression()
    lr.fit(X, s)
    trends_slope = lr.coef_[0]

    return {
        "trends_mean": trends_mean,
        "trends_last30": trends_last30,
        "trends_max": trends_max,
        "trends_slope": trends_slope,
    }

In [ ]:
def normalize_and_score(player_features_df):
    df = player_features_df.copy()

    for col in ["trends_mean", "trends_last30", "trends_max"]:
        df[col] = df[col]/100
    slope_max_abs = np.abs(df["trends_slope"]).max()
    df["trends_slope_norm"] = (df["trends_slope"]/slope_max_abs + 1)/2

    df["final_score"] = (
        0.4 * df["trends_mean"] +
        0.4 * df["trends_last30"] +
        0.1 * df["trends_max"] +
        0.1 * df["trends_slope_norm"]
    )

    return df

In [ ]:
# applied on training set (2019-2023)
training_files = {
    "2019-2020": "processed_normalized_19-20.csv",
    "2020-2021": "processed_normalized_20-21.csv",
    "2021-2022": "processed_normalized_21-22.csv",
    "2022-2023": "processed_normalized_22-23.csv",
}

training_scores = {}

for season, csv in training_files.items():
    df_season = pd.read_csv(csv, index_col=0)
    df_season.index = pd.to_datetime(df_season.index)

    player_features = {player: extract_features(df_season[player]) for player in df_season.columns}
    player_features_df = pd.DataFrame(player_features).T

    scored_df = normalize_and_score(player_features_df)

    final_df = scored_df.reset_index()[["index", "final_score"]]
    final_df.columns = ["player", "final_score"]
    final_df["season"] = season
    training_scores[season] = final_df

training_df = pd.concat(training_scores.values(), ignore_index=True)

#training_df.head(20)

NameError: name 'pd' is not defined

In [ ]:
# applied on test set (2023-2024)
season = "2023-2024"

df_23_24 = pd.read_csv("processed_normalized_23-24.csv", index_col=0)
df_23_24.index = pd.to_datetime(df_23_24.index)

player_features_23_24 = {player: extract_features(df_23_24[player]) for player in df_23_24.columns}
player_features_df_23_24 = pd.DataFrame(player_features_23_24).T

scored_df_23_24 = normalize_and_score(player_features_df_23_24)

test_df = scored_df_23_24.reset_index()[["index", "final_score"]]
test_df.columns = ["player", "final_score"]
test_df["season"] = season

#test_df.head(20)
#test_df.sort_values('final_score', ascending=False).head()

,player,final_score,season
0,Joel Embiid,0.087650,2023-2024
1,Luka Dončić,0.059965,2023-2024
2,Giannis Antetokounmpo,0.096008,2023-2024
3,Shai Gilgeous-Alexander,0.081206,2023-2024
4,LeBron James,0.187950,2023-2024
5,Jalen Brunson,0.081103,2023-2024
6,Devin Booker,0.072813,2023-2024
7,Kevin Durant,0.080290,2023-2024
8,Jayson Tatum,0.087142,2023-2024
9,De'Aaron Fox,0.063161,2023-2024


In [ ]:
#save training and test final scores
training_df.to_csv("google_trends_training_df.csv", index=False)
test_df.to_csv("google_trends_test_df.csv", index=False)

In [ ]:
#apply on season 2025-2026
season = "2025-2026"

df_25_26 = pd.read_csv("processed_normalized_25-26.csv", index_col=0)
df_25_26.index = pd.to_datetime(df_25_26.index)

player_features_25_26 = {player: extract_features(df_25_26[player]) for player in df_25_26.columns}
player_features_df_25_26 = pd.DataFrame(player_features_25_26).T

scored_df_25_26 = normalize_and_score(player_features_df_25_26)

pred_25_26 = scored_df_25_26.reset_index()[["index", "final_score"]]
pred_25_26.columns = ["player", "final_score"]
pred_25_26["season"] = season

pred_25_26.to_csv("google_trends_pred_25_26.csv", index=False)
pred_25_26.sort_values('final_score', ascending=False).head()


,player,final_score,season
4,LeBron James,0.440521,2025-2026
66,Ja Morant,0.232003,2025-2026
78,Cooper Flagg,0.220612,2025-2026
13,Stephen Curry,0.218595,2025-2026
16,Victor Wembanyama,0.195543,2025-2026


### Step 2: Evaluate based on season

1. Metrics used: Precision, Recall, F1, IoU, TP, FP, FN






In [ ]:
def evaluate_predictions(player_features_df, true_allstars, top_k=26, score_col="final_score"):
    ranked = player_features_df.sort_values(score_col, ascending=False)

    top_pred = set(ranked.head(top_k).index)
    true_allstars_set = set(true_allstars)

    tp = len(top_pred & true_allstars_set)
    fp = len(top_pred - true_allstars_set)
    fn = len(true_allstars_set - top_pred)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    iou = tp / len(top_pred | true_allstars_set) if len(top_pred | true_allstars_set) > 0 else 0

    print(f"Top-{top_k} Precision: {precision:.3f}")
    print(f"Top-{top_k} Recall:    {recall:.3f}")
    print(f"Top-{top_k} F1:        {f1:.3f}")
    print(f"Top-{top_k} IoU:       {iou:.3f}")
    print(f"TP: {tp}, FP: {fp}, FN: {fn}")

    return ranked, precision, recall, f1, iou

In [ ]:
#applied on training
for season in training_df["season"].unique():
    print(f"\nSeason: {season}")

    season_df = training_df[training_df["season"] == season].copy()
    season_df.set_index("player", inplace=True)
    true_allstars = all_stars_lists[season]

    ranked, precision, recall, f1, iou = evaluate_predictions(
        season_df,
        true_allstars,
        top_k=26,
        score_col="final_score"
    )
    #print(ranked.head(10))


Season: 2019-2020
Top-26 Precision: 0.423
Top-26 Recall:    0.440
Top-26 F1:        0.431
Top-26 IoU:       0.275
TP: 11, FP: 15, FN: 14
                       final_score     season
player                                       
LeBron James              0.329492  2019-2020
Anthony Davis             0.184857  2019-2020
James Harden              0.166341  2019-2020
Vince Carter              0.164903  2019-2020
Carmelo Anthony           0.156208  2019-2020
Zion Williamson           0.153559  2019-2020
Kemba Walker              0.151560  2019-2020
Tacko Fall                0.148328  2019-2020
Giannis Antetokounmpo     0.144719  2019-2020
Stephen Curry             0.141086  2019-2020

Season: 2020-2021
Top-26 Precision: 0.538
Top-26 Recall:    0.519
Top-26 F1:        0.528
Top-26 IoU:       0.359
TP: 14, FP: 12, FN: 13
               final_score     season
player                               
LeBron James      0.167780  2020-2021
James Harden      0.158344  2020-2021
Anthony Davis     0.

In [ ]:
#applied on test
season = "2023-2024"
season_df = test_df.copy()
season_df.set_index("player", inplace=True)

ranked, precision, recall, f1, iou = evaluate_predictions(
    season_df,
    all_stars_lists[season],
    top_k=26,
    score_col="final_score"
)

#print(ranked.head(10))

Top-26 Precision: 0.462
Top-26 Recall:    0.462
Top-26 F1:        0.462
Top-26 IoU:       0.300
TP: 12, FP: 14, FN: 14
                       final_score     season
player                                       
LeBron James              0.187950  2023-2024
Ja Morant                 0.184808  2023-2024
Draymond Green            0.175624  2023-2024
Josh Giddey               0.149842  2023-2024
OG Anunoby                0.106711  2023-2024
Anthony Edwards           0.106709  2023-2024
RJ Barrett                0.104013  2023-2024
Pascal Siakam             0.098637  2023-2024
Giannis Antetokounmpo     0.096008  2023-2024
Tyrese Haliburton         0.095216  2023-2024
